# Optimal Transport based ICA versus FastICA - linear setting

### We compare the performance of OT based ICA and FastICA over varying number of dimensions and sample size of simulation over a grid.

In [1]:
import numpy as np
import torch
import scipy.stats
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import FastICA
from tqdm.notebook import tqdm

from wasserstein_ica import WassersteinICA

In [2]:
# ==========================================
# 1. Configuration (Grid Search)
# ==========================================
# Dimensions: 2 to 10
DIMENSION_RANGE = list(range(2, 11)) 

# Samples: 100, then 1000 to 10000 in steps of 1000
SAMPLE_SIZE_RANGE = [100] + list(range(1000, 10001, 1000))

# Trials per grid point (Average out noise)
N_TRIALS = 5 

print(f"--- Grid Search Configuration ---")
print(f"Dimensions ({len(DIMENSION_RANGE)}): {DIMENSION_RANGE}")
print(f"Samples ({len(SAMPLE_SIZE_RANGE)}): {SAMPLE_SIZE_RANGE}")
print(f"Total Grid Points: {len(DIMENSION_RANGE) * len(SAMPLE_SIZE_RANGE)}")
print(f"Total Runs: {len(DIMENSION_RANGE) * len(SAMPLE_SIZE_RANGE) * N_TRIALS * 2}")

--- Grid Search Configuration ---
Dimensions (9): [2, 3, 4, 5, 6, 7, 8, 9, 10]
Samples (11): [100, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
Total Grid Points: 99
Total Runs: 990


In [3]:
# ==========================================
# 2. Helpers (Amari & Data Gen)
# ==========================================
def amari_error(W_est, A_true):
    if W_est is None or np.any(np.isnan(W_est)): return np.nan
    P = np.abs(W_est @ A_true)
    n = P.shape[0]
    row_sum = np.sum(P, axis=1)
    row_max = np.max(P, axis=1)
    term1 = np.sum((row_sum / row_max) - 1)
    col_sum = np.sum(P, axis=0)
    col_max = np.max(P, axis=0)
    term2 = np.sum((col_sum / col_max) - 1)
    return (term1 + term2) / (2 * n * (n - 1))

def get_whitening_matrix(X_torch):
    n_samples = X_torch.shape[1]
    X_centered = X_torch - torch.mean(X_torch, dim=1, keepdim=True)
    cov = torch.matmul(X_centered, X_centered.t()) / (n_samples - 1)
    D, E = torch.linalg.eigh(cov)
    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D + 1e-5))
    return torch.matmul(D_inv_sqrt, E.T).cpu().numpy()

def generate_dataset(n_dim, n_samples, seed=None):
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)
    
    sources = []
    # Cycle through 4 distribution types
    for i in range(n_dim):
        dist_type = i % 4
        if dist_type == 0: s = np.random.laplace(0, 1, n_samples)
        elif dist_type == 1: s = np.random.uniform(-1.73, 1.73, n_samples)
        elif dist_type == 2: s = np.random.standard_t(df=3, size=n_samples)
        else: 
            s = np.random.beta(0.5, 0.5, size=n_samples)
            s = (s - np.mean(s)) / np.std(s)
        sources.append(s)
        
    S = np.stack(sources)
    
    # Random Mixing Matrix
    cond_num = 1000
    while cond_num > 100:
        A = np.random.randn(n_dim, n_dim)
        cond_num = np.linalg.cond(A)
        
    X = A @ S
    return torch.tensor(X, dtype=torch.float32), A

In [4]:
# ==========================================
# 3. Execution Loop
# ==========================================
results = []

# Create a progress bar for the total iterations
pbar = tqdm(total=len(DIMENSION_RANGE) * len(SAMPLE_SIZE_RANGE) * N_TRIALS)

for dim in DIMENSION_RANGE:
    for n in SAMPLE_SIZE_RANGE:
        for trial in range(N_TRIALS):
            seed = trial + (dim * 100) + (n * 10) # Unique seed per config
            
            # Generate Data
            X_torch, A_true = generate_dataset(n_dim=dim, n_samples=n, seed=seed)
            X_np = X_torch.numpy()
            
            # --- Method 1: FastICA ---
            try:
                fast_ica = FastICA(n_components=dim, max_iter=2000, tol=1e-3, random_state=seed)
                fast_ica.fit(X_np.T)
                W_fast = fast_ica.components_ #@ fast_ica.whitening_
                score_fast = amari_error(W_fast, A_true)
            except:
                score_fast = np.nan
            
            # --- Method 2: WassersteinICA ---
            try:
                ica = WassersteinICA(X_torch)
                ica.whiten()
                extracted_ws = []
                for _ in range(dim):
                    prev = torch.stack(extracted_ws) if extracted_ws else None
                    # Using the corrected gradient descent logic inside the class
                    w, _ = ica.optimize_wasserstein2(
                        prev_components=prev, max_iter=200, lr=0.1, continuous=True
                    )
                    extracted_ws.append(w)
                
                W_sphere = torch.stack(extracted_ws).cpu().numpy()
                W_white = get_whitening_matrix(X_torch)
                W_wass = W_sphere @ W_white
                score_wass = amari_error(W_wass, A_true)
            except:
                score_wass = np.nan
            
            # Store Results
            results.append({'Dimension': dim, 'Samples': n, 'Method': 'FastICA', 'Amari': score_fast})
            results.append({'Dimension': dim, 'Samples': n, 'Method': 'WassersteinICA', 'Amari': score_wass})
            
            pbar.update(1)

pbar.close()
df_results = pd.DataFrame(results)

  0%|          | 0/495 [00:00<?, ?it/s]

/home/ashutoshjha/py_envs/ot_ica/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/home/ashutoshjha/py_envs/ot_ica/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/home/ashutoshjha/py_envs/ot_ica/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


In [5]:
# ==========================================
# 4. 3D Interactive Surface Plot
# ==========================================

# Aggregate data (Mean over trials)
df_avg = df_results.groupby(['Dimension', 'Samples', 'Method'])['Amari'].mean().reset_index()

# Pivot data for Surface Plot (X=Samples, Y=Dim, Z=Amari)
# We swap X and Y usually for better visual in 3D
pivot_fast = df_avg[df_avg['Method'] == 'FastICA'].pivot(index='Dimension', columns='Samples', values='Amari')
pivot_wass = df_avg[df_avg['Method'] == 'WassersteinICA'].pivot(index='Dimension', columns='Samples', values='Amari')

# X and Y coordinates
x_vals = pivot_fast.columns  # Samples
y_vals = pivot_fast.index    # Dimensions

fig = go.Figure()

# Surface 1: FastICA (Blue-ish)
fig.add_trace(go.Surface(
    z=pivot_fast.values,
    x=x_vals,
    y=y_vals,
    colorscale='Blues',
    opacity=0.8,
    name='FastICA',
    showscale=False,
    colorbar=dict(title='FastICA', x=0)
))

# Surface 2: WassersteinICA (Red/Orange)
fig.add_trace(go.Surface(
    z=pivot_wass.values,
    x=x_vals,
    y=y_vals,
    colorscale='Reds',
    opacity=0.8,
    name='WassersteinICA',
    showscale=True,
    colorbar=dict(title='Amari Error')
))

fig.update_layout(
    title='Amari Error Surface: WassersteinICA (Red) vs FastICA (Blue)',
    scene=dict(
        xaxis_title='Sample Size',
        yaxis_title='Dimensions',
        zaxis_title='Amari Error (Lower is Better)',
        xaxis_type='log',  # Log scale for Sample Size makes it easier to read
        zaxis=dict(range=[0, 0.6]) # Crop huge errors
    ),
    width=900,
    height=700,
)

fig.show()